In [50]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import base64
import requests

# Load API key from .env
load_dotenv()
api_key=os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

def divide_into_batches(array, batch_size):
    no_batches = (len(array) // batch_size) + 1
    batches = []

    for b in range(no_batches):
        batches += [array[b * batch_size : ((b + 1) * batch_size)]]
    
    return batches

In [52]:
path = f'/home/{os.getlogin()}/Pictures/Screenshots/'
image_paths = [path + file_name for file_name in os.listdir(path) if file_name[-4:] == '.png']
already_done = []

with open("already_done.txt", "r", encoding="utf-8") as f:
    already_done += f.read().strip().splitlines()

image_paths = [path for path in image_paths if path not in already_done]
print(image_paths)

with open("already_done.txt", "a", encoding="utf-8") as f:
    for img in image_paths:
        f.write(img + "\n")

base64_images = [encode_image(img_path) for img_path in image_paths]
b64_batches = divide_into_batches(base64_images, batch_size = 3)

[]


In [ ]:
for batch in b64_batches:
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {api_key}"
    }

    content_list = [
        {
            "type": "image_url",
            "image_url": {
            "url": f"data:image/jpeg;base64,{base64_image}"
            }
        } for base64_image in batch
    ]

    content_list = [
            {
                "type": "text",
                "text": "What’s the text in this images?"
            }
    ] + content_list

    payload = {
        "model": "gpt-4.1",
        "messages": [
        {
            "role": "user",
            "content": content_list 
        }
        ],
        "max_tokens": 2000 * len(batch)
    }

    response = requests.post("https://api.openai.com/v1/chat/completions", headers=headers, json=payload)
    print(f"{response.json()}"[:80])
    print("3 images processed")

    response_text = response.json()['choices'][0]['message']['content']
    with open("text.txt", "a", encoding="utf-8") as f:
            f.write(response_text)